# Fine-tuning Whisper Large-V3 — Wolof (281h en 4 phases)

## Comment ca marche :
- **Phase 1** : exemples 0-40K (~80h) — base = whisper-large-v3
- **Phase 2** : exemples 40K-80K (~80h) — base = modele Phase 1
- **Phase 3** : exemples 80K-120K (~80h) — base = modele Phase 2
- **Phase 4** : exemples 120K-144K (~50h) — base = modele Phase 3

## Instructions :
1. Runtime > Change runtime type > **T4 GPU**
2. Change PHASE ci-dessous (1, 2, 3 ou 4)
3. **Ctrl+F9** (Executer tout)
4. Attends 6-8h
5. Pour la phase suivante : change PHASE et relance

---

In [ ]:
#@title 1. CHOISIS TA PHASE (1, 2, 3 ou 4)

PHASE = 1  # <-- CHANGE ICI (1, 2, 3 ou 4)

# Ne touche a rien en dessous
PHASES = {
    1: {"start": 0, "end": 40000, "base": "openai/whisper-large-v3"},
    2: {"start": 40000, "end": 80000, "base": "/content/drive/MyDrive/whisper-wolof-mega"},
    3: {"start": 80000, "end": 120000, "base": "/content/drive/MyDrive/whisper-wolof-mega"},
    4: {"start": 120000, "end": 144314, "base": "/content/drive/MyDrive/whisper-wolof-mega"},
}

config = PHASES[PHASE]
BASE_MODEL = config["base"]
START = config["start"]
END = config["end"]

print(f"{'='*60}")
print(f"   PHASE {PHASE}/4")
print(f"   Exemples: {START:,} a {END:,} ({END-START:,} samples)")
print(f"   Modele de base: {BASE_MODEL}")
print(f"{'='*60}")

In [ ]:
#@title 2. Installation
!pip install -q transformers==4.46.0 datasets==3.1.0 accelerate==1.1.0
!pip install -q evaluate jiwer tensorboard soundfile librosa
!pip install -q huggingface_hub pydub numpy scipy
!apt-get -qq install ffmpeg

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("PAS DE GPU ! -> Runtime > Change runtime type > T4 GPU")

In [ ]:
#@title 3. Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/whisper-wolof-mega"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Sauvegarde dans: {OUTPUT_DIR}")

In [ ]:
#@title 4. Configuration
BATCH_SIZE = 1  # Minimum pour T4 16GB avec Whisper Large
GRADIENT_ACCUMULATION = 32  # Batch effectif = 32
LEARNING_RATE = 5e-6
WARMUP_STEPS = 300
MAX_STEPS = 5000
EVAL_STEPS = 500
SAVE_STEPS = 500
AUGMENT_PROB = 0.4

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"Batch effectif: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"Steps: {MAX_STEPS}")
print(f"LR: {LEARNING_RATE} (cosine)")

In [ ]:
#@title 5. Telecharger SEULEMENT les exemples de cette phase
from datasets import load_dataset, DatasetDict, Audio, concatenate_datasets
import numpy as np
import shutil

print(f"Telechargement exemples {START:,} a {END:,}...")
print(f"(~{(END-START)*7/3600:.0f}h de wolof)")
print(f"")

# Telecharger UNIQUEMENT la tranche necessaire
train_ds = load_dataset(
    "soynade-research/Wolof-ASR-Data",
    split=f"train[{START}:{END}]",
    trust_remote_code=True,
)

test_ds = load_dataset(
    "soynade-research/Wolof-ASR-Data",
    split="test[:3000]",
    trust_remote_code=True,
)

print(f"\nTelecharge: {len(train_ds):,} train, {len(test_ds):,} test")

# Normaliser les colonnes
cols = train_ds.column_names
text_col = next((c for c in ["sentence", "text", "transcription"] if c in cols), None)
audio_col = next((c for c in ["audio", "path", "audio_path"] if c in cols), None)

if text_col and text_col != "sentence":
    train_ds = train_ds.rename_column(text_col, "sentence")
    test_ds = test_ds.rename_column(text_col, "sentence")
if audio_col and audio_col != "audio":
    train_ds = train_ds.rename_column(audio_col, "audio")
    test_ds = test_ds.rename_column(audio_col, "audio")

keep = ["audio", "sentence"]
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in keep])

# Filtrer vides
train_ds = train_ds.filter(lambda x: x["sentence"] and len(x["sentence"].strip()) > 0)
test_ds = test_ds.filter(lambda x: x["sentence"] and len(x["sentence"].strip()) > 0)

# Aussi charger galsenai si Phase 1 (bonus 41h)
if PHASE == 1:
    print(f"\nBonus: galsenai/wolof_tts...")
    try:
        galsen = load_dataset("galsenai/wolof_tts", split="train[:10000]", trust_remote_code=True)
        gcols = galsen.column_names
        gtxt = next((c for c in ["sentence", "text", "transcription"] if c in gcols), None)
        gaud = next((c for c in ["audio", "path", "audio_path"] if c in gcols), None)
        if gtxt and gaud:
            if gtxt != "sentence": galsen = galsen.rename_column(gtxt, "sentence")
            if gaud != "audio": galsen = galsen.rename_column(gaud, "audio")
            galsen = galsen.remove_columns([c for c in galsen.column_names if c not in keep])
            galsen = galsen.cast_column("audio", Audio(sampling_rate=16000))
            train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
            train_ds = concatenate_datasets([train_ds, galsen])
            print(f"   +{len(galsen):,} exemples galsenai")
    except Exception as e:
        print(f"   Pas dispo: {e}")

dataset = DatasetDict({"train": train_ds, "test": test_ds})

# Liberer le cache maintenant
cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir, ignore_errors=True)
    print("\nCache HF libere")

total, used, free = shutil.disk_usage("/")
print(f"Espace libre: {free / 1e9:.1f} GB")
print(f"\nDataset pret: {len(dataset['train']):,} train, {len(dataset['test']):,} test")

In [ ]:
#@title 6. Charger le processeur Whisper
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
)

feature_extractor = WhisperFeatureExtractor.from_pretrained(BASE_MODEL)
tokenizer = WhisperTokenizer.from_pretrained(BASE_MODEL, language="french", task="transcribe")
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="french", task="transcribe")

print("Processor charge")

In [ ]:
#@title 7. Reechantillonner a 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
print("Audio: 16kHz")

In [ ]:
#@title 8. Data Augmentation
import random

def augment_audio(audio_array, sr=16000):
    audio = audio_array.copy().astype(np.float32)
    if random.random() < 0.4:
        audio = audio + np.random.randn(len(audio)).astype(np.float32) * random.uniform(0.002, 0.02)
    if random.random() < 0.3:
        speed = random.uniform(0.88, 1.12)
        indices = np.round(np.arange(0, len(audio), speed)).astype(int)
        indices = indices[indices < len(audio)]
        audio = audio[indices]
    if random.random() < 0.5:
        audio = audio * random.uniform(0.5, 1.5)
    if random.random() < 0.25:
        delay = int(sr * random.uniform(0.02, 0.1))
        decay = random.uniform(0.1, 0.4)
        echo = np.zeros(len(audio) + delay, dtype=np.float32)
        echo[:len(audio)] += audio
        echo[delay:delay+len(audio)] += audio * decay
        audio = echo[:len(audio)]
    if random.random() < 0.35:
        t = np.arange(len(audio)) / sr
        audio = audio + 0.008 * np.sin(2 * np.pi * random.uniform(30, 250) * t).astype(np.float32)
    if random.random() < 0.15:
        try:
            from scipy.signal import butter, filtfilt
            b, a = butter(4, [300/(sr/2), 3400/(sr/2)], btype='band')
            audio = filtfilt(b, a, audio).astype(np.float32)
        except:
            pass
    mx = np.max(np.abs(audio))
    if mx > 0:
        audio = audio / mx * 0.95
    return audio.astype(np.float32)

print("Augmentation prete")

In [ ]:
#@title 9. Dataset on-the-fly (ZERO cache disque)
import torch.utils.data

class WhisperDataset(torch.utils.data.Dataset):
    """Preprocesse audio a la volee — pas de fichier cache sur disque."""
    def __init__(self, hf_dataset, feature_extractor, tokenizer, augment=False):
        self.dataset = hf_dataset
        self.feature_extractor = feature_extractor
        self.tokenizer = tokenizer
        self.augment = augment

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        audio = item["audio"]
        audio_array = np.array(audio["array"], dtype=np.float32)
        sr = audio["sampling_rate"]

        if self.augment and random.random() < AUGMENT_PROB:
            audio_array = augment_audio(audio_array, sr)

        input_features = self.feature_extractor(
            audio_array, sampling_rate=sr
        ).input_features[0]

        labels = self.tokenizer(item["sentence"]).input_ids

        return {"input_features": input_features, "labels": labels}

train_dataset = WhisperDataset(dataset["train"], feature_extractor, tokenizer, augment=True)
test_dataset = WhisperDataset(dataset["test"], feature_extractor, tokenizer, augment=False)

total, used, free = shutil.disk_usage("/")
print(f"Espace libre: {free / 1e9:.1f} GB")
print(f"Train: {len(train_dataset):,} | Test: {len(test_dataset):,}")
print("Mode on-the-fly: ZERO espace disque supplementaire !")

In [ ]:
#@title 10. Charger le modele
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)

model.generation_config.language = "fr"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False

print(f"Modele: {BASE_MODEL}")
print(f"Params: {model.num_parameters() / 1e6:.0f}M")

In [ ]:
#@title 11. Preparer l'entrainement
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>"),
)

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("OK")

In [ ]:
#@title 12. LANCER L'ENTRAINEMENT
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-wolof-mega",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    remove_unused_columns=False,
    label_names=["labels"],
    push_to_hub=False,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print(f"\n{'='*60}")
print(f"   PHASE {PHASE} - ENTRAINEMENT")
print(f"   {MAX_STEPS} steps (~{MAX_STEPS*3/3600:.0f}-{MAX_STEPS*5/3600:.0f}h)")
print(f"   NE FERMEZ PAS CETTE PAGE !")
print(f"{'='*60}\n")

trainer.train()

In [ ]:
#@title 13. Evaluation
results = trainer.evaluate()
print(f"\nWER Phase {PHASE}: {results['eval_wer']:.1f}%")
print(f"(< 40% correct, < 25% bon, < 15% excellent)")

In [ ]:
#@title 14. Sauvegarder sur Google Drive
save_local = "./whisper-wolof-mega-final"
trainer.save_model(save_local)
processor.save_pretrained(save_local)
tokenizer.save_pretrained(save_local)

!cp -r {save_local}/* {OUTPUT_DIR}/

print(f"\n{'='*60}")
print(f"   PHASE {PHASE} TERMINEE !")
print(f"   Modele sauvegarde: {OUTPUT_DIR}")
print(f"{'='*60}")

if PHASE < 4:
    print(f"\n   PROCHAINE ETAPE:")
    print(f"   1. Change PHASE = {PHASE + 1} dans la cellule 1")
    print(f"   2. Deconnecte la session (Execution > Gerer les sessions > Supprimer)")
    print(f"   3. Reconnecte et Ctrl+F9")
    print(f"   Le modele Phase {PHASE} sera utilise comme base pour Phase {PHASE + 1}")
else:
    print(f"\n   TOUTES LES PHASES TERMINEES !")
    print(f"   Telechargez {OUTPUT_DIR} depuis Drive")
    print(f"   Placez dans: wolof-transcriber/backend/models/whisper-wolof-mega/")
    print(f"   Lancez python app.py")

In [ ]:
#@title 15. (Optionnel) Tester le modele
from transformers import pipeline as hf_pipeline

pipe = hf_pipeline(
    "automatic-speech-recognition",
    model=save_local,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device="cuda",
    chunk_length_s=30,
    batch_size=8,
)

print(f"Test Phase {PHASE}:")
print("="*60)

for i in range(min(5, len(test_ds))):
    sample = test_ds[i]
    result = pipe(sample["audio"]["array"])
    print(f"\n--- {i+1} ---")
    print(f"  REF: {sample['sentence']}")
    print(f"  OUT: {result['text']}")